# EDA: Аудиоданные для классификации дорожного покрытия

Разведочный анализ аудио данных проекта Road Surface Classification.

## Содержание
1. [Загрузка данных](#Загрузка-данных)
2. [Статистика датасета](#Статистика-датасета)
3. [Визуализация аудиосигналов](#Визуализация-аудиосигналов)
4. [Спектральный анализ](#Спектральный-анализ)
5. [Анализ по классам](#Анализ-по-классам)

In [ ]:
# Импорт библиотек
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchaudio
import librosa
import librosa.display

# Добавляем проект в path
sys.path.insert(0, str(Path.cwd().parent))

# Настройка стилей
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
%matplotlib inline

## Загрузка данных

In [ ]:
# Пути к данным
DATA_ROOT = Path("../data/raw")

# Классы дорожного покрытия
CLASS_NAMES = ["asphalt", "concrete", "gravel", "dirt", "snow"]
CLASS_MAPPING = {
    "asphalt": "Асфальт",
    "concrete": "Бетон",
    "gravel": "Гравий",
    "dirt": "Грунт",
    "snow": "Снег",
}

In [ ]:
# Сканирование данных
def scan_audio_files(data_root: Path) -> pd.DataFrame:
    """Scan directory for audio files."""
    records = []
    
    for class_name in CLASS_NAMES:
        class_dir = data_root / class_name
        if not class_dir.exists():
            continue
            
        for audio_file in class_dir.glob("*.wav"):
            records.append({
                "path": str(audio_file),
                "filename": audio_file.name,
                "class": class_name,
                "class_ru": CLASS_MAPPING[class_name],
            })
    
    return pd.DataFrame(records)

df = scan_audio_files(DATA_ROOT)
print(f"Найдено файлов: {len(df)}")

## Статистика датасета

In [ ]:
# Распределение по классам
class_counts = df["class"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(range(len(class_counts)), class_counts.values)
plt.xticks(range(len(class_counts)), [CLASS_MAPPING[c] for c in class_counts.index], rotation=45)
plt.ylabel("Количество файлов")
plt.title("Распределение аудиофайлов по классам дорожного покрытия")
plt.tight_layout()
plt.show()

print("\nСтатистика:")
print(f"  Всего файлов: {len(df)}")
print(f"  Классов: {len(class_counts)}")
print(f"  Среднее на класс: {class_counts.mean():.1f}")

## Визуализация аудиосигналов

In [ ]:
def load_audio_sample(path: str, duration: float = 5.0) -> tuple[np.ndarray, int]:
    """Load audio sample with optional duration limit."""
    waveform, sample_rate = torchaudio.load(path)
    
    # Limit duration
    max_samples = int(duration * sample_rate)
    if waveform.shape[1] > max_samples:
        waveform = waveform[:, :max_samples]
    
    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    return waveform.squeeze().numpy(), sample_rate


def plot_waveform(waveform: np.ndarray, sample_rate: int, title: str = "") -> None:
    """Plot audio waveform."""
    time_axis = np.arange(len(waveform)) / sample_rate
    
    plt.figure(figsize=(14, 4))
    plt.plot(time_axis, waveform, linewidth=0.5)
    plt.xlabel("Время (с)")
    plt.ylabel("Амплитуда")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Примеры для каждого класса
fig, axes = plt.subplots(len(CLASS_NAMES), 1, figsize=(14, 3 * len(CLASS_NAMES)))

for idx, class_name in enumerate(CLASS_NAMES):
    class_files = df[df["class"] == class_name]["path"].tolist()
    if not class_files:
        continue
    
    # Берем случайный файл
    sample_path = np.random.choice(class_files)
    waveform, sample_rate = load_audio_sample(sample_path, duration=3.0)
    
    time_axis = np.arange(len(waveform)) / sample_rate
    axes[idx].plot(time_axis, waveform, linewidth=0.5)
    axes[idx].set_ylabel(CLASS_MAPPING[class_name])
    axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel("Время (с)")
plt.suptitle("Примеры аудиосигналов для каждого класса", fontsize=14)
plt.tight_layout()
plt.show()

## Спектральный анализ

In [ ]:
def plot_spectrogram(waveform: np.ndarray, sample_rate: int, title: str = "") -> None:
    """Plot mel spectrogram."""
    # Вычисление mel-спектрограммы
    spec = librosa.feature.melspectrogram(
        y=waveform,
        sr=sample_rate,
        n_mels=128,
        n_fft=2048,
        hop_length=512,
    )
    
    # Конвертация в dB
    spec_db = librosa.power_to_db(spec, ref=np.max)
    
    plt.figure(figsize=(14, 5))
    librosa.display.specshow(
        spec_db,
        sr=sample_rate,
        hop_length=512,
        x_axis="time",
        y_axis="mel",
    )
    plt.colorbar(format="%+2.0f dB")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    
    return spec_db

In [ ]:
# Спектрограммы для каждого класса
for class_name in CLASS_NAMES:
    class_files = df[df["class"] == class_name]["path"].tolist()
    if not class_files:
        continue
    
    sample_path = np.random.choice(class_files)
    waveform, sample_rate = load_audio_sample(sample_path, duration=5.0)
    
    plot_spectrogram(
        waveform,
        sample_rate,
        title=f"Mel-спектрограмма: {CLASS_MAPPING[class_name]}",
    )

## Анализ по классам

In [ ]:
def extract_audio_features(path: str) -> dict:
    """Extract basic audio features."""
    waveform, sample_rate = load_audio_sample(path, duration=5.0)
    
    # Duration
    duration = len(waveform) / sample_rate
    
    # RMS energy
    rms = np.sqrt(np.mean(waveform ** 2))
    
    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(waveform)[0].mean()
    
    # Spectral centroid
    centroid = librosa.feature.spectral_centroid(
        y=waveform, sr=sample_rate
    )[0].mean()
    
    return {
        "duration": duration,
        "rms": rms,
        "zcr": zcr,
        "spectral_centroid": centroid,
    }

In [ ]:
# Извлечение признаков для выборки файлов
sample_size = min(100, len(df))
sample_df = df.sample(n=sample_size, random_state=42)

features = []
for path in sample_df["path"]:
    try:
        feat = extract_audio_features(path)
        feat["class"] = sample_df[sample_df["path"] == path]["class"].values[0]
        features.append(feat)
    except Exception as e:
        print(f"Error processing {path}: {e}")

features_df = pd.DataFrame(features)

In [ ]:
# Визуализация признаков по классам
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_to_plot = [
    ("rms", "RMS энергия"),
    ("zcr", "Zero Crossing Rate"),
    ("spectral_centroid", "Спектральный центр (Гц)"),
    ("duration", "Длительность (с)"),
]

for ax, (feature, title) in zip(axes.flat, features_to_plot):
    sns.boxplot(
        data=features_df,
        x="class",
        y=feature,
        ax=ax,
    )
    ax.set_xticklabels([CLASS_MAPPING[c] for c in features_df["class"].unique()], rotation=45)
    ax.set_ylabel(title)
    ax.set_xlabel("")

plt.suptitle("Распределение аудио-признаков по классам", fontsize=14)
plt.tight_layout()
plt.show()

## Выводы

1. **Баланс классов**: Проверить равномерность распределения файлов по классам
2. **Характерные признаки**: Какие аудио-признаки лучше всего разделяют классы
3. **Качество данных**: Наличие шумов, артефактов записи
4. **Спектральные паттерны**: Визуальные различия на спектрограммах